## 1. Setup & Parse Arguments

In [1]:
# Auto-reload modules
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from argparse import Namespace
import sys
from datetime import datetime
import gc
import torch
from train import train

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Ti
GPU Memory: 17.18 GB


In [2]:
from config import (
    BATCH_SIZE, NUM_EPOCHS, LEARNING_RATE,
    MODE, FUSION_METHOD, FEATURE_MODE,
    CHECKPOINT_DIR, BEST_MODEL_NAME,
)

# ============================================================================
# CẤU HÌNH GRID SEARCH CHO DỰ ÁN DUAL-STREAM (FBANK + HANDCRAFTED)
# ============================================================================
ROOT_DIR = r"E:\speech_data\features\train" # Thư mục gốc chứa các thư mục duration
durations = ['train_raw', 'train_vi_full']  # 'train_raw', 'train_vi_3s', 'train_vi_5s', 'train_vi_7s', 'train_vi_full'

# Các feature mode (Dựa vào config.py của bạn)
feature_modes = ["mfbe_pitch", "mfbe_only", "pitch_only"]
fusion_methods = ["concat", "cross_attention", "gating"]

def get_base_args():
    return Namespace(
        test_dir="D:/test", # Thay đổi nếu cần
        output_dir="./outputs",
        batch_size=64, # Dự án này dùng Conv1D nhỏ nên 64-128 là đẹp
        learning_rate=0.0001,
        lr_scheduler="plateau",
        num_epochs=50,
        optimizer="adam",
        weight_decay=0.001,
        mixed_precision=True,
        early_stop_patience=10,
        seed=42,
        device="cuda",
        
        # Biến gán động
        mode=3,
        exp_name="",
        base_dir="",
        feature_mode="",
        fusion_method="",
        duration=""
    )

def check_skip(exp_name):
    if os.path.exists(os.path.join("./outputs/experiments", exp_name, "results.json")):
        print(f"⏭ Bỏ qua: {exp_name} - Đã xong!")
        return True
    return False

## 2. Training

In [3]:
# 1. MODE 1: CHỈ DÙNG FBANK (5 Experiments)
for dur in durations:
    exp_name = f"Mode1_FBank_{dur}"
    if check_skip(exp_name): continue
    
    print(f"\n{'='*80}\nĐANG CHẠY: {exp_name}\n{'='*80}")
    args = get_base_args()
    args.mode = 1
    args.exp_name = exp_name
    args.duration = dur
    args.base_dir = os.path.join(ROOT_DIR, dur)
    args.feature_mode = "N/A"
    args.fusion_method = "N/A"
    
    train(args)
    gc.collect(); torch.cuda.empty_cache()

# 2. MODE 2: CHỈ DÙNG HANDCRAFTED (15 Experiments)
for dur in durations:
    for feat in feature_modes:
        exp_name = f"Mode2_HC_{dur}_{feat}"
        if check_skip(exp_name): continue
        
        print(f"\n{'='*80}\nĐANG CHẠY: {exp_name}\n{'='*80}")
        args = get_base_args()
        args.mode = 2
        args.exp_name = exp_name
        args.duration = dur
        args.base_dir = os.path.join(ROOT_DIR, dur)
        args.feature_mode = feat
        args.fusion_method = "N/A"
        
        train(args)
        gc.collect(); torch.cuda.empty_cache()

# 3. MODE 3: HYBRID FUSION (45 Experiments)
for dur in durations:
    for feat in feature_modes:
        for fusion in fusion_methods:
            exp_name = f"Mode3_{fusion}_{dur}_{feat}"
            if check_skip(exp_name): continue
            
            print(f"\n{'='*80}\nĐANG CHẠY: {exp_name}\n{'='*80}")
            args = get_base_args()
            args.mode = 3
            args.exp_name = exp_name
            args.duration = dur
            args.base_dir = os.path.join(ROOT_DIR, dur)
            args.feature_mode = feat
            args.fusion_method = fusion
            
            train(args)
            gc.collect(); torch.cuda.empty_cache()

print("\n🎉 TOÀN BỘ GRID SEARCH ĐÃ HOÀN TẤT THÀNH CÔNG!")


ĐANG CHẠY: Mode1_FBank_train_raw
🔍 Đang quét dữ liệu Train/Val tại: E:\speech_data\features\train\train_raw...
  -> Tối ưu RAM: Đang nạp data từ fbank_shards...

Khởi tạo mô hình thành công (Hybrid ECAPA-TDNN)
  Mode: 1 (1=Main, 2=Handcrafted, 3=Fusion)
  Num speakers: 2380
  Trainable parameters: 2,640,896

Starting training...



Epoch   1 | Train Loss: 6.7297, Acc: 0.2604 | Val Loss: 7.5920, Acc: 0.1387 | LR: 0.000100


Epoch   2 | Train Loss: 4.2652, Acc: 0.4046 | Val Loss: 6.9964, Acc: 0.1611 | LR: 0.000100


Epoch   3 | Train Loss: 3.5203, Acc: 0.4674 | Val Loss: 6.9356, Acc: 0.1617 | LR: 0.000100


Epoch   4 | Train Loss: 3.1789, Acc: 0.4998 | Val Loss: 7.1675, Acc: 0.1339 | LR: 0.000100


...

Epoch  47 | Train Loss: 5.0599, Acc: 0.4141 | Val Loss: 2.6191, Acc: 0.5763 | LR: 0.000025


Epoch  48 | Train Loss: 5.0534, Acc: 0.4148 | Val Loss: 2.5563, Acc: 0.5889 | LR: 0.000025


Epoch  49 | Train Loss: 5.0507, Acc: 0.4145 | Val Loss: 2.6948, Acc: 0.5619 | LR: 0.000025


Epoch  50 | Train Loss: 5.0489, Acc: 0.4151 | Val Loss: 2.6288, Acc: 0.5741 | LR: 0.000025

ĐANG CHẠY: Mode2_HC_train_raw_mfbe_only
🔍 Đang quét dữ liệu Train/Val tại: E:\speech_data\features\train\train_raw...
  -> Tối ưu RAM: Đang nạp data từ mfbe_shards...

Khởi tạo mô hình thành công (Hybrid ECAPA-TDNN)
  Mode: 2 (1=Main, 2=Handcrafted, 3=Fusion)
  Num speakers: 2380
  Trainable parameters: 1,051,136

Starting training...



RuntimeError: Given groups=1, weight of size [128, 81, 3], expected input[64, 80, 300] to have 81 channels, but got 80 channels instead

## 3. Training Curves

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Chạy giao diện TensorBoard trỏ vào thư mục chứa tất cả các experiments
%tensorboard --logdir ./outputs/experiments

## 4. Test Results

In [ ]:
import os
import json
from datetime import datetime
from train import load_checkpoint
from inference import evaluate_speaker_verification
from dataset import create_test_loader

# 1. Load Data
print("Loading Independent Test Data (Unseen Speakers)...")
test_loader, test_num_speakers = create_test_loader(
    test_dir=args.test_dir, 
    mode=args.mode, 
    feature_mode=args.feature_mode, 
    batch_size=args.batch_size, 
    num_workers=0
)
print(f"✓ Loaded {test_num_speakers} unseen speakers for testing")

# 2. Load Best Model từ exp_dir
best_model_path = os.path.join(exp_dir, "best_model.pth")
model, _, _, _ = load_checkpoint(best_model_path, model)
model = model.to(device)

# 3. Chạy Đánh giá EER & MinDCF
test_results = evaluate_speaker_verification(
    model=model, 
    data_loader=test_loader, 
    device=device,
    num_pairs=50000,
    p_target=0.05
)

# In kết quả ra màn hình
print("\n" + "="*50)
print("OPEN-SET BENCHMARK RESULTS (TEST SET)")
print("="*50)
for k, v in test_results.items():
    print(f"{k:<25}: {v:.4f}")
print("="*50)

# 4. TẠO FILE test_results.json VỚI TOÀN BỘ THÔNG SỐ
test_results_file = os.path.join(exp_dir, "test_results.json")

# Sử dụng vars(args) để lấy 100% các thông số cấu hình đã khai báo
final_test_data = {
    "test_timestamp": datetime.now().isoformat(),
    "config": vars(args),  # <-- Tự động lấy toàn bộ args (batch_size, lr, aam_margin, mode...)
    "metrics": test_results
}

with open(test_results_file, "w") as f:
    json.dump(final_test_data, f, indent=4)
    
print(f"✓ Đã lưu kết quả Test cùng TOÀN BỘ thông số cấu hình vào: {test_results_file}")

## 5. Gating Analysis (Only for Mode 3 + Gating)

In [ ]:
from train import analyze_gating_behavior
import matplotlib.image as mpimg

# Kiểm tra điều kiện để chạy phân tích
if args.mode == 3 and args.fusion_method == "gating":
    print("Đang thực hiện phân tích cơ chế Gating trên tập Test...")
    
    # Gọi hàm phân tích từ train.py
    # Lưu ý: Hàm này trả về 2 giá trị: gates (trọng số PTM) và labels
    gates, labels = analyze_gating_behavior(model, test_loader, device, exp_dir)
    
    # Hiển thị biểu đồ phân phối trọng số đã được lưu
    gate_plot_path = os.path.join(exp_dir, "gating_analysis", "gate_distribution.png")
    if os.path.exists(gate_plot_path):
        plt.figure(figsize=(10, 6))
        img = mpimg.imread(gate_plot_path)
        plt.imshow(img)
        plt.axis('off')
        plt.title("Phân phối trọng số Gating (Trục X > 0.5 ưu tiên PTM, < 0.5 ưu tiên Handcrafted)")
        plt.show()
        
    # In thông tin thống kê tóm tắt
    ptm_wins = np.sum(gates > 0.5)
    hc_wins = np.sum(gates <= 0.5)
    print(f"Thống kê nhanh:")
    print(f"   - Số lần ưu tiên PTM (WavLM/HuBERT): {ptm_wins} ({100*ptm_wins/len(gates):.2f}%)")
    print(f"   - Số lần ưu tiên Handcrafted (MFCC/Pitch): {hc_wins} ({100*hc_wins/len(gates):.2f}%)")
else:
    print("ℹChế độ hiện tại không sử dụng Gating. Bỏ qua bước phân tích này.")

## 6. Experiment Comparison

In [ ]:
import pandas as pd

# List all experiments
exp_base_dir = "./outputs/experiments"
if os.path.exists(exp_base_dir):
    experiments = []
    for exp_name_dir in sorted(os.listdir(exp_base_dir)):
        exp_path = os.path.join(exp_base_dir, exp_name_dir)
        results_file = os.path.join(exp_path, "results.json")
        if os.path.exists(results_file):
            with open(results_file, "r") as f:
                data = json.load(f)
                config = data.get("config", {})
                experiments.append({
                    "Experiment": exp_name_dir,
                    "Mode": config.get("mode", ""),
                    "Fusion": config.get("fusion_method", "N/A"),
                    "Feature": config.get("feature_mode", "N/A"),
                    "Best Val Loss": f"{data.get('best_val_loss', 0):.4f}",
                    "Epochs": data.get("epochs_trained", 0),
                })
    
    if experiments:
        df = pd.DataFrame(experiments)
        print("\n" + "="*120)
        print("EXPERIMENT COMPARISON")
        print("="*120)
        print(df.to_string(index=False))
        print("="*120)
    else:
        print("No experiments found.")
else:
    print(f"Directory {exp_base_dir} does not exist yet.")